# Chapter 13 — Claude Code Built-in Tools · Practice

**Goal:** practice choosing and using Claude Code's built-in tools the way a careful agent does: Glob for paths, Grep for contents, Read only what matters, Edit with unique anchors (and the Read + Write fallback when that is impossible), Bash for running things, permission rules per tool, and an incremental investigation of an unfamiliar repository. The mini-project is a small offline investigation agent that traces a refund flow through a wrapper module.

- **Theory:** [README.md](README.md) (read section 13.x before the matching section here)
- **Estimated time:** 2.5–3.5 hours
- **Estimated API cost:** **$0** for everything except the single optional live cell at the end, which runs the Claude Agent SDK with `max_budget_usd=0.30`: the run stops once the SDK's client-side cost *estimate* reaches $0.30, and the response that crosses the threshold still counts, so the final cost can slightly exceed it. The real cost has not been measured for this notebook; expect a few cents with the default model, `claude-sonnet-5-5`, and check the printed `total_cost_usd`.
- **Prerequisites:** [00-prerequisites](../../00-prerequisites/README.md) (venv from `requirements.txt`), [Chapter 2](../02-tool-use/README.md) (tool definitions), [Chapter 3](../03-agent-sdk/README.md) (`query()`, `ClaudeAgentOptions`), [Chapter 5](../05-claude-code/README.md) (plan mode, Explore subagent)

**How this notebook works**

| Marker | Meaning |
|---|---|
| **(offline)** | Runs without a key or network. The cells use `FakeRepo`: an in-memory copy of a small legacy repository (`acme-shop`) with simulated `Glob`, `Grep`, `Read`, `Write`, `Edit` and `Bash` tools that follow the contracts in the [tools reference](https://code.claude.com/docs/en/tools-reference). Every call is logged with a rough token cost, so you can *measure* what a strategy costs in context. |
| **(live, optional)** | One cell at the very end. It runs only when you set `RUN_LIVE = True` in the setup cell **and** `ANTHROPIC_API_KEY` is in `.env`. It starts the Claude Agent SDK in a temporary copy of the repository, restricted to `Read`, `Grep` and `Glob`, with `max_budget_usd` set. |

The simulator is a teaching model: its error texts are its own, and the real tools have more features (images, PDFs, sandboxing, hooks). Where a detail matters for the exam, the README cites the docs.

In [ ]:
# Setup (offline)
import json
import os
import re
import tempfile
import textwrap
from collections import Counter
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")

RUN_LIVE = False   # set to True to run the one optional live Agent SDK cell at the end (needs ANTHROPIC_API_KEY)


def require_live() -> None:
    """Guard for the live cell: never start the Agent SDK without an explicit API key."""
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError("Live cell skipped: set ANTHROPIC_API_KEY in .env (see 00-prerequisites).")


print("MODEL =", MODEL, "| RUN_LIVE =", RUN_LIVE)

In [ ]:
# Key check: only needed for the live cell. Offline cells never need a key.
if RUN_LIVE:
    assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env (see 00-prerequisites)"
    print("API key found: the live cell is enabled.")
else:
    print("Offline mode: every cell except the optional live one runs without a key.")

### The simulator (run, don't edit)

`FakeRepo.call(tool, **input)` takes the same input field names as the real tools (`pattern`, `path`, `glob`, `type`, `output_mode`, `file_path`, `old_string`, `new_string`, `replace_all`, `command`, ...) and returns a `ToolResult(tool, input, ok, output)`. `repo.ledger` keeps every call; `repo.context_tokens()` adds up their rough cost (about 4 characters per token).

In [ ]:
# (offline) Simulated built-in tools
# (offline) A fake repository with simulated Claude Code built-in tools.
# The behavior follows the documented contracts in the tools reference (code.claude.com/docs/en/tools-reference):
# Glob matches paths (sorted by modification time, capped at 100, ignores .gitignore); Grep searches contents
# (regex, three output modes, respects .gitignore unless `path` names the file itself); Read returns numbered lines and a PARTIAL view for big files;
# Write overwrites whole files; Edit needs read-before-edit, an exact match and a unique match (or replace_all).
# Read-before-edit/overwrite is modeled as always required, as on Opus 4.6, Haiku 4.5 and older models (and the exam);
# newer models may skip it when reading would not prompt, but notebooks and PARTIAL views always need the read.
# Error texts are written for this simulator; the real tool's wording differs.
import fnmatch
import re
from dataclasses import dataclass, field

GLOB_CAP = 100
TYPE_EXTS = {"py": (".py",), "ts": (".ts", ".tsx"), "js": (".js", ".jsx"), "md": (".md",),
             "yaml": (".yaml", ".yml"), "json": (".json",), "toml": (".toml",)}


@dataclass
class ToolResult:
    tool: str
    input: dict
    ok: bool
    output: str

    @property
    def tokens(self) -> int:
        """Rough context cost of this result (about 4 characters per token)."""
        return max(1, len(self.output) // 4)

    def __repr__(self) -> str:
        head = self.output if len(self.output) < 160 else self.output[:157] + "..."
        return f"{self.tool}{'' if self.ok else ' [ERROR]'} {self.input} -> {head!r}"


def glob_to_regex(pattern: str) -> re.Pattern:
    """Glob syntax: ** spans directories, * and ? stay inside one segment, {a,b} alternates."""
    i, out = 0, []
    while i < len(pattern):
        c = pattern[i]
        if pattern.startswith("**/", i):
            out.append("(?:.*/)?"); i += 3; continue
        if pattern.startswith("**", i):
            out.append(".*"); i += 2; continue
        if c == "*":
            out.append("[^/]*")
        elif c == "?":
            out.append("[^/]")
        elif c == "{":
            j = pattern.index("}", i)
            out.append("(?:" + "|".join(re.escape(p) for p in pattern[i + 1:j].split(",")) + ")"); i = j
        else:
            out.append(re.escape(c))
        i += 1
    return re.compile("".join(out) + r"\Z")


class FakeRepo:
    """An in-memory repository plus the six core tools. Every call is logged in self.ledger."""

    def __init__(self, files: dict[str, str], gitignore: tuple[str, ...] = (), read_token_limit: int = 2500,
                 git_log: dict[str, int] | None = None):
        self.files = dict(files)
        self.mtime = {p: i for i, p in enumerate(self.files)}
        self.clock = len(self.files)
        self.gitignore = tuple(gitignore)
        self.read_token_limit = read_token_limit
        self.git_log = dict(git_log or {})      # path -> number of commits in the last 12 months
        self.read_at: dict[str, int] = {}       # path -> mtime of the version Claude has fully seen
        self.ledger: list[ToolResult] = []

    # ---------- helpers ----------
    def _tick(self, path: str) -> None:
        self.clock += 1
        self.mtime[path] = self.clock

    def _ignored(self, path: str) -> bool:
        parts = path.split("/")
        for pat in self.gitignore:
            if pat.endswith("/"):
                if pat.rstrip("/") in parts[:-1]:
                    return True
            elif any(fnmatch.fnmatch(seg, pat) for seg in parts):
                return True
        return False

    def _under(self, path: str, base: str | None) -> bool:
        base = (base or "").strip("/")
        return not base or path == base or path.startswith(base + "/")

    def is_dir(self, path: str) -> bool:
        p = path.strip("/")
        return p == "" or any(f.startswith(p + "/") for f in self.files)

    def external_change(self, path: str, content: str) -> None:
        """Simulate a teammate or formatter changing a file on disk (not logged as a tool call)."""
        self.files[path] = content
        self._tick(path)

    def context_tokens(self) -> int:
        return sum(r.tokens for r in self.ledger)

    def calls(self, tool: str | None = None) -> list[ToolResult]:
        return [r for r in self.ledger if tool is None or r.tool == tool]

    def reset_ledger(self) -> None:
        self.ledger.clear()

    # ---------- dispatcher ----------
    def call(self, tool: str, **inp) -> ToolResult:
        handler = {"Glob": self._glob, "Grep": self._grep, "Read": self._read, "Write": self._write,
                   "Edit": self._edit, "Bash": self._bash}.get(tool)
        if handler is None:
            res = ToolResult(tool, inp, False, f"Error: no such tool in this simulator: {tool}")
        else:
            try:
                ok, out = handler(**inp)
            except TypeError as exc:                     # wrong or missing input fields
                ok, out = False, f"InputValidationError: {exc}"
            res = ToolResult(tool, inp, ok, out)
        self.ledger.append(res)
        return res

    def read_raw(self, path: str) -> str | None:
        """Read through the Read tool (logged), then return the raw file text without line numbers."""
        r = self.call("Read", file_path=path)
        if not r.ok or "PARTIAL view" in r.output:
            return None
        return self.files[path]

    # ---------- tools ----------
    def _glob(self, pattern: str, path: str | None = None):
        rx = glob_to_regex(pattern)
        base = (path or "").strip("/")
        hits = [f for f in self.files if self._under(f, base) and rx.match(f[len(base) + 1:] if base else f)]
        hits.sort(key=lambda f: -self.mtime[f])          # newest first in this simulator
        truncated = len(hits) > GLOB_CAP
        lines = hits[:GLOB_CAP]
        if truncated:
            lines.append(f"(Results are truncated: showing {GLOB_CAP} of {len(hits)}. Use a more specific pattern.)")
        return True, "\n".join(lines) if lines else "No files found"

    def _grep(self, pattern: str, path: str | None = None, glob: str | None = None, type: str | None = None,
              output_mode: str = "files_with_matches", head_limit: int | None = None, offset: int = 0,
              multiline: bool = False, **flags):
        if output_mode not in ("files_with_matches", "content", "count"):
            return False, f"Error: invalid output_mode {output_mode!r}"
        re_flags = (re.IGNORECASE if flags.get("-i") else 0) | (re.MULTILINE | re.DOTALL if multiline else 0)
        try:
            rx = re.compile(pattern, re_flags)
        except re.error as exc:
            return False, f"Error: regex parse error in {pattern!r}: {exc}"
        if glob:
            g = glob_to_regex(glob if "/" in glob else "**/" + glob)
        exts = TYPE_EXTS.get(type, ()) if type else ()
        if type and not exts:
            return False, f"Error: unrecognized file type: {type}"
        explicit = (path or "").strip("/")                 # a path naming one file is searched even if gitignored
        files = [f for f in sorted(self.files) if self._under(f, path) and (f == explicit or not self._ignored(f))
                 and (not glob or g.match(f)) and (not exts or f.endswith(exts))]
        before, after = flags.get("-B", flags.get("-C", 0)) or 0, flags.get("-A", flags.get("-C", 0)) or 0
        entries, counts = [], {}
        for f in files:
            text = self.files[f]
            if multiline:
                n = len(rx.findall(text))
                if n:
                    counts[f] = n
                    if output_mode == "content":
                        for m in rx.finditer(text):
                            line_no = text.count("\n", 0, m.start()) + 1
                            entries.append(f"{f}:{line_no}:{m.group(0)}")
                continue
            lines = text.splitlines()
            hit_lines = [i for i, ln in enumerate(lines) if rx.search(ln)]
            if not hit_lines:
                continue
            counts[f] = len(hit_lines)
            if output_mode == "content":
                shown = sorted({j for i in hit_lines for j in range(max(0, i - before), min(len(lines), i + after + 1))})
                entries += [f"{f}:{j + 1}:{lines[j]}" if j in hit_lines else f"{f}-{j + 1}-{lines[j]}" for j in shown]
        if output_mode == "files_with_matches":
            entries = list(counts)
        elif output_mode == "count":
            entries = [f"{f}:{n}" for f, n in counts.items()]
        total = len(entries)
        page = entries[offset: offset + head_limit if head_limit else None]
        if not page:
            return True, "No matches found" if total == 0 else "No entries at this offset"
        out = "\n".join(page)
        if output_mode == "count":
            out += f"\nTotal: {sum(counts.values())} matches in {len(counts)} files"
        elif len(page) < total:
            out += f"\n(showing {len(page)} of {total}; use offset to see more)"
        return True, out

    def _read(self, file_path: str, offset: int | None = None, limit: int | None = None):
        if self.is_dir(file_path) and file_path not in self.files:
            return False, f"Error: {file_path} is a directory. Read only reads files; list directories with ls via Bash."
        if file_path not in self.files:
            return False, f"Error: file does not exist: {file_path}"
        lines = self.files[file_path].splitlines()
        if not lines:
            self.read_at[file_path] = self.mtime[file_path]
            return True, "(the file exists but its contents are empty)"
        start = (offset or 1) - 1
        if start >= len(lines):
            return True, f"(offset is past the end: the file has {len(lines)} lines)"
        end = len(lines) if limit is None else min(len(lines), start + limit)
        body = "\n".join(f"{i + 1:>6}\t{lines[i]}" for i in range(start, end))
        if offset is None and limit is None and len(body) // 4 > self.read_token_limit:
            cut, size = [], 0
            for i, ln in enumerate(lines):
                row = f"{i + 1:>6}\t{ln}"
                if (size + len(row)) // 4 > self.read_token_limit:
                    break
                cut.append(row); size += len(row) + 1
            return True, "\n".join(cut) + (f"\n[PARTIAL view: showing lines 1-{len(cut)} of {len(lines)}. "
                                           f"Read more with offset and limit, or Grep for what you need.]")
        self.read_at[file_path] = self.mtime[file_path]
        return True, body

    def _fresh(self, path: str) -> bool:
        return self.read_at.get(path) == self.mtime.get(path)

    def _write(self, file_path: str, content: str):
        if file_path in self.files and file_path not in self.read_at:
            return False, "Error: file has not been read yet. Read it before overwriting it."
        existed = file_path in self.files
        self.files[file_path] = content
        self._tick(file_path)
        self.read_at[file_path] = self.mtime[file_path]
        return True, f"{'Overwrote' if existed else 'Created'} {file_path} ({len(content.splitlines())} lines)"

    def _edit(self, file_path: str, old_string: str, new_string: str, replace_all: bool = False):
        if file_path not in self.files:
            return False, f"Error: file does not exist: {file_path}"
        if file_path not in self.read_at:
            return False, "Error: file has not been read yet. Read it first before editing it."
        if old_string == new_string:
            return False, "Error: old_string and new_string are identical; nothing to change."
        text = self.files[file_path]
        n = text.count(old_string)
        stale = not self._fresh(file_path)
        if n == 0:
            hint = " The file changed after you read it; read it again." if stale else \
                " Check whitespace and indentation; copy the text exactly from a fresh Read."
            return False, "Error: string to replace not found in file." + hint
        if n > 1 and not replace_all:
            return False, (f"Error: found {n} matches of the string to replace, but replace_all is false. "
                           "To replace all occurrences, set replace_all to true. To replace only one, "
                           "provide more surrounding context so old_string matches exactly once.")
        self.files[file_path] = text.replace(old_string, new_string) if replace_all else text.replace(old_string, new_string, 1)
        self._tick(file_path)
        self.read_at[file_path] = self.mtime[file_path]
        note = " Note: the file had changed since your last read; re-read it before edits that depend on context." if stale else ""
        return True, f"Edited {file_path}: replaced {n if replace_all else 1} occurrence(s).{note}"

    def _bash(self, command: str, timeout: int | None = None, description: str | None = None,
              run_in_background: bool = False):
        cmd = command.strip()
        if cmd == "ls" or cmd.startswith("ls "):
            base = cmd[2:].strip().strip("/")
            if base and not self.is_dir(base):
                return False, f"ls: {base}: No such file or directory"
            names = sorted({f[len(base) + 1 if base else 0:].split("/")[0] + ("/" if "/" in f[len(base) + 1 if base else 0:] else "")
                            for f in self.files if self._under(f, base)})
            return True, "\n".join(names)
        m = re.fullmatch(r"(cat|head|tail) (\S+)", cmd)
        if m:   # a plain single-file view also satisfies read-before-edit
            path = m.group(2)
            if path not in self.files:
                return False, f"{m.group(1)}: {path}: No such file or directory"
            self.read_at[path] = self.mtime[path]
            lines = self.files[path].splitlines()
            lines = lines[:10] if m.group(1) == "head" else lines[-10:] if m.group(1) == "tail" else lines
            return True, "\n".join(lines)
        m = re.fullmatch(r"wc -l (\S+)", cmd)
        if m and m.group(1) in self.files:
            return True, f"{len(self.files[m.group(1)].splitlines())} {m.group(1)}"
        if cmd.startswith("git log") and "--name-only" in cmd:
            return True, "\n".join(f"{n:>4} commits  {p}" for p, n in sorted(self.git_log.items(), key=lambda kv: -kv[1]))
        if cmd.startswith(("pytest", "python -m pytest")):
            tests = [f for f in self.files if re.search(r"(^|/)test_[^/]*\.py$", f)]
            return True, f"{'.' * len(tests)}\n{len(tests)} passed in 0.42s (simulated)"
        if cmd.startswith(("git diff", "git status")):
            return True, "(simulated: nothing to show)"
        return False, f"(simulated shell) command not supported here: {cmd}"

### The repository: `acme-shop`

A Python backend (`app/`) with orders, billing, payments, support macros and a nightly worker; a React front end (`web/`) with 110 generated icon components; tests; deployment files; a gitignored `build/` folder; and one large generated file. `app/billing/__init__.py` is a **wrapper module** that re-exports `create_refund` as `issue_refund`.

In [ ]:
# (offline) Fixture: the acme-shop repository
# (offline) acme-shop: a small "legacy" repository used by every exercise in this chapter.
ACME_FILES: dict[str, str] = {
    "README.md": "# acme-shop\n\nOrders, billing and refunds for the Acme web shop. Python backend in app/, React front end in web/.\n",
    "pyproject.toml": "[project]\nname = \"acme-shop\"\nversion = \"3.4.1\"\ndependencies = [\"flask>=3\", \"requests\"]\n",
    ".gitignore": "build/\n*.log\nnode_modules/\n",
    "app/__init__.py": "",
    "app/api/__init__.py": "",
    "app/api/routes.py": '''from app.billing import issue_refund
from app.orders.service import get_order
from app.web import route


@route("/orders/<order_id>/refund", methods=["POST"])
def refund_order(order_id: str, amount: float):
    order = get_order(order_id)
    return issue_refund(order.id, amount)


@route("/orders/<order_id>", methods=["GET"])
def show_order(order_id: str):
    return get_order(order_id)
''',
    "app/web.py": '''def route(path, methods=("GET",)):
    def wrap(fn):
        fn.route = (path, methods)
        return fn
    return wrap
''',
    "app/orders/__init__.py": "",
    "app/orders/service.py": '''from app.orders.models import Order


def get_order(order_id: str) -> Order:
    return Order.load(order_id)


def cancel_order(order_id: str) -> None:
    Order.load(order_id).cancel()
''',
    "app/orders/models.py": '''class Order:
    def __init__(self, id: str, total: float):
        self.id, self.total = id, total

    @classmethod
    def load(cls, order_id: str) -> "Order":
        return cls(order_id, 120.0)

    def cancel(self) -> None:
        self.status = "cancelled"
''',
    "app/billing/__init__.py": '''"""Public billing API. Import from here, not from the submodules."""
from .refunds import create_refund as issue_refund, cancel_refund
from .ledger import post_entry

__all__ = ["issue_refund", "cancel_refund", "post_entry"]
''',
    "app/billing/refunds.py": '''from app.billing.ledger import post_entry
from app.billing.policy import check_policy
from app.payments.gateway import Gateway


def create_refund(order_id: str, amount: float) -> dict:
    check_policy(order_id, amount)
    receipt = Gateway().submit_refund(order_id, amount)
    post_entry(order_id, -amount, memo="refund")
    return {"order_id": order_id, "amount": amount, "receipt": receipt}


def cancel_refund(refund_id: str) -> None:
    post_entry(refund_id, 0.0, memo="refund cancelled")
''',
    "app/billing/ledger.py": '''ACCOUNTS = {"closed": set()}


def post_entry(account: str, amount: float, memo: str = "") -> None:
    if account in ACCOUNTS["closed"]:
        raise ValueError("Ledger entry rejected: account closed")
    print(f"ledger {account} {amount:+.2f} {memo}")
''',
    "app/billing/policy.py": '''MAX_REFUND = 500.0


def check_policy(order_id: str, amount: float) -> None:
    if amount > MAX_REFUND:
        raise PermissionError(f"Refund of {amount} exceeds the {MAX_REFUND} limit; escalate to a supervisor")
''',
    "app/payments/__init__.py": "",
    "app/payments/gateway.py": '''import requests


class GatewayError(Exception):
    pass


class Gateway:
    BASE_URL = "https://payments.example.com/v2"

    def charge(self, order_id: str, amount: float) -> str:
        r = requests.post(f"{self.BASE_URL}/charges", json={"order": order_id, "amount": amount}, timeout=30)
        return r.json()["id"]

    def submit_refund(self, order_id: str, amount: float) -> str:
        r = requests.post(f"{self.BASE_URL}/refunds", json={"order": order_id, "amount": amount}, timeout=30)
        if r.status_code == 402:
            raise GatewayError("Refund declined by processor")
        return r.json()["id"]
''',
    "app/support/__init__.py": "",
    "app/support/macros.py": '''import app.billing as billing


def goodwill_refund(order_id: str) -> dict:
    """Support macro: refund 10 per cent of an order as a goodwill gesture."""
    return billing.issue_refund(order_id, 12.0)
''',
    "app/workers/__init__.py": "",
    "app/workers/nightly.py": '''from app.billing import cancel_refund, issue_refund as refund
from app.billing import post_entry


def retry_failed_refunds(failed: list[tuple[str, float]]) -> None:
    for order_id, amount in failed:
        refund(order_id, amount)


def close_day(totals: dict[str, float]) -> None:
    for account, amount in totals.items():
        post_entry(account, amount, memo="end of day")


def expire_pending(refund_ids: list[str]) -> None:
    for rid in refund_ids:
        cancel_refund(rid)
''',
    "app/config/settings.py": '''SERVICES = {
    "payments": {
        "url": "https://payments.example.com/v2",
        "timeout": 30,
        "retries": 3,
    },
    "ledger": {
        "url": "https://ledger.internal/v1",
        "timeout": 30,
        "retries": 3,
    },
    "search": {
        "url": "https://search.internal/v1",
        "timeout": 10,
        "retries": 1,
    },
}
''',
    "deploy/services.yaml": '''services:
  - name: api
    resources:
      replicas: 2
      maxSurge: 1
  - name: billing
    resources:
      replicas: 2
      maxSurge: 1
  - name: worker
    resources:
      replicas: 2
      maxSurge: 1
''',
    "deploy/nginx.conf": """server {
    listen 80;
    server_name shop.example.com;

    location /api/ {
        proxy_pass http://api:8000;
        proxy_read_timeout 30s;
    }
}

server {
    listen 443 ssl;
    server_name shop.example.com;

    location /api/ {
        proxy_pass http://api:8000;
        proxy_read_timeout 30s;
    }
}
""",
    "legacy/old_refunds.py": '''# Deprecated in 2019; kept for reference. Not imported anywhere.
def create_refund(order_id, amount):
    raise NotImplementedError("use app.billing")
''',
    "tests/test_refunds.py": '''from app.billing.refunds import create_refund


def test_small_refund_ok():
    assert create_refund("A-1", 10.0)["amount"] == 10.0
''',
    "tests/test_ledger.py": '''from app.billing.ledger import post_entry


def test_post_entry_prints(capsys):
    post_entry("A-1", 5.0)
    assert "A-1" in capsys.readouterr().out
''',
    "build/lib/app/payments/gateway.py": '# build artefact (gitignored)\nraise GatewayError("Refund declined by processor")\n',
    "web/package.json": '{"name": "acme-web", "scripts": {"test": "vitest", "lint": "eslint src"}}\n',
    "web/src/components/RefundButton.tsx": "export function RefundButton() { return <button>Refund</button>; }\n",
    "web/src/components/RefundButton.test.tsx": "import { RefundButton } from './RefundButton';\ntest('renders', () => {});\n",
    "web/src/components/OrderList.tsx": "export function OrderList() { return <ul />; }\n",
    "web/src/components/OrderList.test.tsx": "import { OrderList } from './OrderList';\ntest('renders', () => {});\n",
    "web/src/pages/orders/OrderPage.tsx": "import { OrderList } from '../../components/OrderList';\nexport default function OrderPage() { return <OrderList />; }\n",
    "web/src/pages/orders/OrderPage.test.tsx": "import OrderPage from './OrderPage';\ntest('renders', () => {});\n",
    "notebooks/refund_analysis.ipynb": '{"cells": [], "metadata": {}, "nbformat": 4, "nbformat_minor": 5}\n',
}

# 110 generated icon components: enough to hit the 100-result Glob cap
for _i in range(110):
    ACME_FILES[f"web/src/icons/Icon{_i:03d}.tsx"] = f"export const Icon{_i:03d} = () => <svg data-id='{_i}' />;\n"

# One big generated file: enough to trigger Read's PARTIAL view
ACME_FILES["app/generated/rates_table.py"] = "RATES = {\n" + "".join(
    f"    \"{c}{n:03d}\": {1 + n / 1000:.3f},\n" for c in "ABCDEFGH" for n in range(160)) + "}\n"

ACME_GITIGNORE = ("build/", "*.log", "node_modules/")

# Commits per file in the last 12 months (what `git log --name-only` would tell you)
ACME_GIT_LOG = {
    "app/billing/refunds.py": 41, "app/payments/gateway.py": 23, "app/api/routes.py": 17,
    "app/billing/ledger.py": 9, "app/orders/service.py": 6, "app/billing/policy.py": 12,
    "app/workers/nightly.py": 14, "app/support/macros.py": 3, "app/orders/models.py": 2,
}

In [ ]:
# (offline) A fresh repository and two small printing helpers
def new_repo() -> FakeRepo:
    return FakeRepo(ACME_FILES, ACME_GITIGNORE, git_log=ACME_GIT_LOG)


def show(res: ToolResult, max_lines: int = 12) -> None:
    lines = res.output.splitlines()
    status = "ok" if res.ok else "ERROR"
    print(f"── {res.tool}({json.dumps(res.input)[:110]}) [{status}, ~{res.tokens} tokens]")
    for ln in lines[:max_lines]:
        print("   ", ln)
    if len(lines) > max_lines:
        print(f"    ... ({len(lines) - max_lines} more lines)")


repo = new_repo()
print(len(repo.files), "files;", sum(f.endswith(".py") for f in repo.files), "Python files")
show(repo.call("Bash", command="ls"))

## 13.1 The six core tools and how Claude chooses among them

Each built-in tool is an ordinary tool definition, and Claude picks one by matching the request against the descriptions. The useful mental model: **paths → Glob, contents → Grep, the code of a known file → Read, a new file or full rewrite → Write, a targeted change → Edit, running something → Bash.** Below, a tiny keyword router makes those selection criteria explicit. It is a teaching device, not how Claude works internally. [Theory](README.md#131-the-six-core-tools-and-how-claude-chooses-among-them)

In [ ]:
# (offline) The selection criteria as data
TOOL_GUIDE = {
    "Glob":  "find files by PATH pattern (names, extensions, directories)",
    "Grep":  "search file CONTENTS with a regex (symbols, error text, imports, config keys)",
    "Read":  "load the contents of a file you already identified",
    "Write": "create a new file, or overwrite a file completely",
    "Edit":  "replace an exact, unique snippet in an existing file",
    "Bash":  "run a command: tests, builds, git, package managers, ls",
}

PATH_HINT = re.compile(r"(\*|\.\w{1,5}\b.*\bfiles?\b|\bfiles? (named|matching|ending)|\bwhich files exist\b)")
CONTENT_HINT = re.compile(r"\b(where|who calls|callers?|uses?|usages?|references?|imports?|mentions?|error message|contains?)\b")


def route(request: str) -> tuple[str, str]:
    """Return (tool, reason) for a developer request. Order matters: the most specific signal wins."""
    r = request.lower()
    quoted = re.findall(r"[`'\"]([^`'\"]+)[`'\"]", request)
    if re.search(r"\b(run|execute|install|build|commit|push|pytest|npm|lint)\b", r):
        return "Bash", "it asks to RUN something"
    if re.search(r"\b(create|new file|scaffold|add a file)\b", r):
        return "Write", "it creates a new file"
    if re.search(r"\b(change|rename|replace|bump|set .* to)\b", r) and re.search(r"\bin [\w/.]+\.\w+", r):
        return "Edit", "a targeted change in a known file"
    if re.search(r"\b(show|open|read)\b", r) and re.search(r"[\w/]+\.\w{1,5}\b", request) and "*" not in request:
        return "Read", "the file is already known"
    if quoted and "*" in quoted[0] or PATH_HINT.search(r):
        return "Glob", "a PATH pattern"
    if CONTENT_HINT.search(r) or quoted:
        return "Grep", "a CONTENT search"
    return "Grep", "default: search before you read"


REQUESTS = [
    ("List every test file matching '**/*.test.tsx'", "Glob"),
    ("Where is the error message 'Refund declined by processor' raised?", "Grep"),
    ("Who calls `issue_refund`?", "Grep"),
    ("Show me app/billing/policy.py", "Read"),
    ("Create a new file app/billing/limits.py with the refund limits", "Write"),
    ("Change the payments timeout to 45 in app/config/settings.py", "Edit"),
    ("Run the test suite with pytest", "Bash"),
    ("Which modules import requests?", "Grep"),
    ("Which files exist under deploy/ ending in .yaml files", "Glob"),
    ("How does a refund work?", "Grep"),
]
for req, expected in REQUESTS:
    tool, why = route(req)
    print(f"{'✓' if tool == expected else '✗'} {tool:5} <- {req}   ({why})")

> **Observe:** the last request has no file name and no exact string, so the router falls back to "search before you read". That is also what a good agent does: it looks for an entry point (a route, a function name) instead of opening files at random.

> **Try it:** add three requests of your own, including one that is ambiguous ("find the refund tests"). Is it a path question (Glob `tests/**/test_*refund*.py`) or a content question (Grep `def test_.*refund`)? Both can be right; decide which is cheaper.

## 13.2 Grep vs Glob: content search vs path search

Glob matches **paths** (sorted by modification time, capped at 100, does not respect `.gitignore`). Grep searches **contents** with ripgrep-style regex (three output modes, `type` / `glob` filters, respects `.gitignore`). Choosing the wrong one gives you either nothing or far too much. [Theory](README.md#132-grep-vs-glob-content-search-vs-path-search)

In [ ]:
# (offline) Glob: path patterns
repo = new_repo()
show(repo.call("Glob", pattern="**/*.test.tsx"))
show(repo.call("Glob", pattern="*.{toml,md}"))            # no "/" and no "**": top level only
show(repo.call("Glob", pattern="**/*.tsx"), max_lines=3)   # 116 matches: capped at 100, with a truncation note
show(repo.call("Glob", pattern="*.tsx", path="web/src/components"))

> **Observe:** the `**/*.tsx` result says it is truncated. An agent that treats the first 100 results as "all of them" will miss files. Narrow the pattern (`web/src/components/**/*.tsx`) instead.

In [ ]:
# (offline) The .gitignore difference: Glob sees the build artefact, Grep skips it unless you name the file
repo = new_repo()
show(repo.call("Glob", pattern="**/gateway.py"))
show(repo.call("Grep", pattern="Refund declined by processor"))
show(repo.call("Grep", pattern="Refund declined by processor", path="build/lib/app/payments/gateway.py"))
print("\n.gitignore:", ACME_GITIGNORE)

In [ ]:
# (offline) Grep output modes: start small, then zoom in
repo = new_repo()
for mode in ("files_with_matches", "count", "content"):
    show(repo.call("Grep", pattern=r"\bpost_entry\b", type="py", output_mode=mode), max_lines=8)
show(repo.call("Grep", pattern=r"def submit_refund", output_mode="content", **{"-A": 4}))

In [ ]:
# (offline) Regex pitfalls and filters
repo = new_repo()
show(repo.call("Grep", pattern="submit_refund(", type="py"))            # unescaped "(" is a regex error
show(repo.call("Grep", pattern=r"submit_refund\(", type="py"))          # escaped: finds the definition AND the call
show(repo.call("Grep", pattern=r"\.submit_refund\(", type="py", output_mode="content"))   # calls only
show(repo.call("Grep", pattern="refund", glob="*.test.tsx", **{"-i": True}))              # glob filter + case-insensitive
# multiline: a POST route decorator followed by its handler
show(repo.call("Grep", pattern=r'@route\([^)]*"POST"[^)]*\)\s*\ndef (\w+)', multiline=True, output_mode="content"))

> **Observe:** the cost column. `files_with_matches` returns a few short paths; `content` returns whole lines. Asking for paths first and content second keeps the context small on large repositories.

> **Try it:** find every Python file that imports `requests` (Grep `^import requests|^from requests`), then find every YAML file (Glob). Which call would you get wrong if you swapped the tools?

## 13.3 Read, Write and Edit: full-file vs targeted changes

Read returns numbered lines (the numbers are not part of the file) and a **`PARTIAL view`** for files over the token limit. Write creates or **completely overwrites** a file, and an existing file must be read first. Edit replaces an **exact, unique** string in a file you have read. (The simulator, like the exam, always requires that read; in current Claude Code, newer models such as Sonnet 5.5 may skip it when reading would not prompt, while Opus 4.6, Haiku 4.5 and older models, notebooks and `PARTIAL view` files always need it.) NotebookEdit edits a notebook cell by `cell_id`. [Theory](README.md#133-read-write-and-edit-full-file-vs-targeted-changes)

In [ ]:
# (offline) Read: line numbers, PARTIAL view, ranges, directories
repo = new_repo()
show(repo.call("Read", file_path="app/billing/policy.py"))
big = repo.call("Read", file_path="app/generated/rates_table.py")
print("\nPARTIAL view?", "PARTIAL view" in big.output, "| tokens:", big.tokens)
print(big.output.splitlines()[-1])
show(repo.call("Read", file_path="app/generated/rates_table.py", offset=1000, limit=3))
show(repo.call("Read", file_path="app/billing"))

In [ ]:
# (offline) Write: new files are fine; overwriting needs a prior Read
repo = new_repo()
show(repo.call("Write", file_path="app/billing/limits.py", content="DAILY_REFUND_LIMIT = 2000.0\n"))
show(repo.call("Write", file_path="app/billing/policy.py", content="MAX_REFUND = 750.0\n"))   # not read yet
repo.call("Read", file_path="app/billing/policy.py")
show(repo.call("Write", file_path="app/billing/policy.py", content="MAX_REFUND = 750.0\n"))
print("\ncheck_policy still defined?", "def check_policy" in repo.files["app/billing/policy.py"])

> **Observe:** the last Write "succeeded" and silently deleted `check_policy`, because Write replaces the whole file with exactly what you pass. That is why partial changes go through Edit, and why the Read + Write fallback (13.4) always works on the *full* current content.

In [ ]:
# (offline) Edit: the happy path, plus what a NotebookEdit call looks like
repo = new_repo()
repo.call("Read", file_path="app/billing/policy.py")
show(repo.call("Edit", file_path="app/billing/policy.py", old_string="MAX_REFUND = 500.0", new_string="MAX_REFUND = 750.0"))
print(repo.files["app/billing/policy.py"].splitlines()[0], "| check_policy kept:", "def check_policy" in repo.files["app/billing/policy.py"])

# NotebookEdit targets one cell by id (the simulator does not implement it; this is the real input shape)
notebook_edit_call = {"notebook_path": "notebooks/refund_analysis.ipynb", "cell_id": "a1b2c3",
                      "new_source": "df.groupby('reason').amount.sum()", "edit_mode": "replace"}
print("NotebookEdit input:", notebook_edit_call)

## 13.4 When Edit fails: anchors, `replace_all` and the Read + Write fallback

Edit failures are deterministic, so retrying the same call never helps. Diagnose the failed check (not read, no match, not unique), then climb the ladder: re-read and copy exactly → `replace_all` if every occurrence should change → widen the anchor with unique context → **Read + Write** when no small unique anchor exists, then verify. [Theory](README.md#134-when-edit-fails-anchors-replace_all-and-the-read--write-fallback)

In [ ]:
# (offline) Each failure mode once
repo = new_repo()
cfg = "app/config/settings.py"
show(repo.call("Edit", file_path=cfg, old_string='"timeout": 30', new_string='"timeout": 45'))       # 1. not read
repo.call("Read", file_path=cfg)
show(repo.call("Edit", file_path=cfg, old_string='"timeout":30', new_string='"timeout": 45'))        # 2. no match (spacing)
show(repo.call("Edit", file_path=cfg, old_string='"timeout": 30', new_string='"timeout": 45'))       # 3. not unique (2 matches)
show(repo.call("Edit", file_path=cfg, old_string='"timeout": 30', new_string='"timeout": 45'))       # 4. retrying changes nothing

In [ ]:
# (offline) Fixes: widen the anchor (one service) or replace_all (every service)
repo = new_repo()
cfg = "app/config/settings.py"
repo.call("Read", file_path=cfg)
anchor_old = '"ledger": {\n        "url": "https://ledger.internal/v1",\n        "timeout": 30,'
anchor_new = '"ledger": {\n        "url": "https://ledger.internal/v1",\n        "timeout": 45,'
show(repo.call("Edit", file_path=cfg, old_string=anchor_old, new_string=anchor_new))
print(re.findall(r'"timeout": \d+', repo.files[cfg]))

other = new_repo()
other.call("Read", file_path=cfg)
show(other.call("Edit", file_path=cfg, old_string='"retries": 3', new_string='"retries": 5', replace_all=True))
print(re.findall(r'"retries": \d+', other.files[cfg]))

Now a file where the repeated block is identical and the only difference (`listen 443 ssl;`) sits several lines above it. You *can* build an anchor that spans from `listen 443` down to the timeout, but it is long and fragile. The reliable alternative is the fallback: **Read the full file, change the right occurrence by structure, Write it back, verify.**

In [ ]:
# (offline) The Read + Write fallback, done carefully
def rewrite_block(repo: FakeRepo, path: str, select, old: str, new: str, split: str = r"(?m)(?=^\S)") -> str:
    """Read the full current file, cut it into blocks where the regex `split` matches (default: a block starts at
    every line that begins in column 0), replace `old` with `new` in the ONE block `select(block)` accepts, Write
    the result, and verify. Refuses to write unless exactly one block is selected and `old` occurs once in it."""
    text = repo.read_raw(path)                                   # 1. Read (logged) the full, current content
    if text is None:
        return "could not read the full file"
    blocks = re.split(split, text)                               # blocks that match the file's structure
    targets = [i for i, b in enumerate(blocks) if select(b) and old in b]
    if len(targets) != 1:
        return f"expected exactly one target block, found {len(targets)}: not writing"
    i = targets[0]
    if blocks[i].count(old) != 1:                                # the block is too coarse for this file
        return f"`old` occurs {blocks[i].count(old)} times in the selected block: not writing (use a finer `split`)"
    blocks[i] = blocks[i].replace(old, new)                      # 2. modify the content in code
    updated = "".join(blocks)
    res = repo.call("Write", file_path=path, content=updated)    # 3. Write the whole file
    if not res.ok:
        return res.output
    after = repo.read_raw(path)                                  # 4. verify
    changed = sum(a != b for a, b in zip(text.splitlines(), after.splitlines()))
    return f"rewrote {path}: {changed} line(s) changed"


repo = new_repo()
conf = "deploy/nginx.conf"
repo.call("Read", file_path=conf)
show(repo.call("Edit", file_path=conf, old_string="proxy_read_timeout 30s;", new_string="proxy_read_timeout 60s;"))
print(rewrite_block(repo, conf, select=lambda b: "listen 443" in b,
                    old="proxy_read_timeout 30s;", new="proxy_read_timeout 60s;"))
print(repo.files[conf])
print("Tool calls:", [r.tool for r in repo.ledger])

# The block boundaries must match the file's structure. In YAML every service is indented, so the default
# column-0 split sees the whole file as ONE block; the uniqueness check refuses instead of changing every service.
svc = "deploy/services.yaml"
repo.call("Read", file_path=svc)
print(rewrite_block(repo, svc, select=lambda b: "- name: billing" in b, old="replicas: 2", new="replicas: 4"))
print(rewrite_block(repo, svc, select=lambda b: "- name: billing" in b, old="replicas: 2", new="replicas: 4",
                    split=r"(?m)(?=^  - name: )"))                # one block per service
print(re.findall(r"name: (\w+)\n\s+resources:\n\s+replicas: (\d+)", repo.files[svc]))   # only billing changed

In [ ]:
# (offline) A stale read: someone changed the file after Claude read it
repo = new_repo()
pol = "app/billing/policy.py"
repo.call("Read", file_path=pol)
repo.external_change(pol, repo.files[pol].replace("MAX_REFUND = 500.0", "MAX_REFUND = 600.0"))   # a teammate's commit
show(repo.call("Edit", file_path=pol, old_string="MAX_REFUND = 500.0", new_string="MAX_REFUND = 750.0"))
show(repo.call("Edit", file_path=pol, old_string="escalate to a supervisor", new_string="escalate to the refunds team"))

> **Observe:** the first Edit fails because its `old_string` came from the stale read. The second one still applies, because its text matches the current file exactly and uniquely, and the result warns that the file changed. Claude Code behaves the same way: matching against current content keeps the edit safe, and Claude re-reads before edits that depend on surrounding context.

> **Try it:** in a fresh repository, change `replicas` in `deploy/services.yaml` to 4 for the `billing` service only, this time with a widened Edit anchor (include `- name: billing`), and check that `api` and `worker` keep `replicas: 2`. Which approach would you trust in a file with 40 services, and why does `rewrite_block` need a `split` that matches the file's structure?

## 13.5 Bash: the general-purpose tool and its limits

Bash runs tests, builds, `git` and anything else without a dedicated tool. Each command is a separate process (exports do not persist), output is bounded, and some exit codes of 1 are normal ("no matches"). A plain single-file `cat` / `head` / `tail` counts as reading the file for Edit. Prefer Read and Edit for files: they carry line numbers, checks and path permissions. [Theory](README.md#135-bash-the-general-purpose-tool-and-its-limits)

In [ ]:
# (offline) Simulated shell
repo = new_repo()
show(repo.call("Bash", command="ls app/billing"))
show(repo.call("Bash", command="git log --name-only --since=12.months"), max_lines=5)
show(repo.call("Bash", command="pytest -q"))
show(repo.call("Bash", command="cat app/billing/policy.py"), max_lines=3)
show(repo.call("Edit", file_path="app/billing/policy.py", old_string="MAX_REFUND = 500.0", new_string="MAX_REFUND = 450.0"))
show(repo.call("Bash", command="sed -i 's/500.0/450.0/' app/billing/policy.py"))

In [ ]:
# (offline) Why a noisy command is a context problem: the shape of a Bash result
INLINE_OK, INLINE_FAIL, PREVIEW = 30_000, 10_000, 2_000   # approximate limits from the tools reference


def bash_result_view(output: str, failed: bool) -> str:
    if not failed:
        if len(output) <= INLINE_OK:
            return output
        return f"[output saved to a file: {len(output):,} chars]\n" + output[:PREVIEW]
    if len(output) <= INLINE_FAIL:
        return output
    half = INLINE_FAIL // 2
    return output[:half] + "\n[... middle omitted ...]\n" + output[-half:]


log = "\n".join(f"test_case_{i:04d} PASSED" for i in range(4000))
view = bash_result_view(log, failed=False)
print(f"full log: {len(log):,} chars (~{len(log) // 4:,} tokens) -> what Claude sees: {len(view):,} chars")
print(view.splitlines()[0])

> **Observe:** even the truncated view of a 4,000-test log is thousands of tokens of noise. Running the suite in a subagent (or with `-q` and a failure filter) and returning only the failures is the context-friendly pattern.

## 13.6 Incremental investigation: Grep, Read, repeat

Question: *"Customers see 'Refund declined by processor'. What code path produces it, and where is the refund amount checked?"* Compare an incremental investigation with "read every Python file first". [Theory](README.md#136-incremental-investigation-grep-read-repeat)

In [ ]:
# (offline) The incremental way: every step is chosen from the previous result
repo = new_repo()
steps = [
    ("Grep", dict(pattern="Refund declined by processor", output_mode="content")),
    ("Read", dict(file_path="app/payments/gateway.py")),
    ("Grep", dict(pattern=r"\.submit_refund\(", type="py", output_mode="content")),
    ("Read", dict(file_path="app/billing/refunds.py")),
    ("Grep", dict(pattern=r"def check_policy", type="py", output_mode="content")),
    ("Read", dict(file_path="app/billing/policy.py")),
]
for tool, inp in steps:
    show(repo.call(tool, **inp), max_lines=4)
incremental = repo.context_tokens()
print(f"\nIncremental: {len(repo.ledger)} calls, {len(repo.calls('Read'))} files read, ~{incremental:,} tokens")

In [ ]:
# (offline) The "read everything first" way
repo = new_repo()
py_files = repo.call("Glob", pattern="**/*.py").output.splitlines()
for f in py_files:
    if f.endswith(".py"):
        repo.call("Read", file_path=f)
read_all = repo.context_tokens()
print(f"Read-all: {len(repo.ledger)} calls, {len(repo.calls('Read'))} files read, ~{read_all:,} tokens "
      f"({read_all / incremental:.1f}x the incremental cost)")
partial = [r.input["file_path"] for r in repo.calls("Read") if "PARTIAL view" in r.output]
print("Files only partially seen:", partial)

> **Observe:** this toy repository has only 23 Python files, and "read everything" already costs several times more context (about 9x here), while the answer is no better. In a real 2,000-file repository it does not fit at all, and the relevant lines end up buried in the middle of a huge context.

> **Try it:** write the investigation for *"Where does the nightly job post ledger entries, and what happens if the account is closed?"* Start from a Grep and keep it under five calls.

**Keep a scratchpad.** For longer investigations, write confirmed findings to a file as you go and read it back before the next phase (task 5.4):

In [ ]:
# (offline) A scratchpad of confirmed findings (file:line evidence only, no guesses)
repo = new_repo()
hit = repo.call("Grep", pattern="Refund declined by processor", output_mode="content").output.splitlines()[0]
path, line, _ = hit.split(":", 2)
notes = textwrap.dedent(f"""\
    # investigation-scratchpad.md
    ## Confirmed findings
    - Error text raised in {path}:{line} (Gateway.submit_refund, HTTP 402 from the processor)
    - Gateway.submit_refund is called from app/billing/refunds.py (create_refund)
    ## Open questions
    - Which public entry points reach create_refund? (it is re-exported; see 13.7)
""")
show(repo.call("Write", file_path="notes/investigation-scratchpad.md", content=notes))

## 13.7 Tracing through wrapper modules

`app/billing/__init__.py` re-exports `create_refund` **as** `issue_refund`. Searching only for the definition name misses almost every caller. The method: list every exported name (aliases included), then search for each one. [Theory](README.md#137-tracing-through-wrapper-modules)

In [ ]:
# (offline) The naive search
repo = new_repo()
show(repo.call("Grep", pattern=r"\bcreate_refund\b", type="py", output_mode="content"))

In [ ]:
# (offline) The wrapper-aware search
IMPORT_LINE = re.compile(r"^from\s+\.?[\w.]*\s+import\s+(.+)$", re.M)


def exported_names(init_source: str) -> dict[str, str]:
    """Map exported name -> original name for 'from .x import a as b, c' lines (single-line form only)."""
    names = {}
    for group in IMPORT_LINE.findall(init_source):
        for item in group.split(","):
            parts = item.strip().split(" as ")
            original, exported = parts[0].strip(), parts[-1].strip()
            names[exported] = original
    return names


repo = new_repo()
wrapper = "app/billing/__init__.py"
src = repo.read_raw(wrapper)
exports = exported_names(src)
print("exports:", exports)
callers = {}
for exported in exports:
    res = repo.call("Grep", pattern=rf"\b{re.escape(exported)}\b", type="py")
    callers[exported] = [f for f in res.output.splitlines() if f.endswith(".py") and not f.startswith("app/billing/")]
for name, files in callers.items():
    print(f"{name:14} (defined as {exports[name]}): {files}")

In [ ]:
# (offline) Second-level aliases: the nightly worker renames issue_refund again
repo = new_repo()
show(repo.call("Grep", pattern=r"import .*\bissue_refund as (\w+)", type="py", output_mode="content"))
show(repo.call("Grep", pattern=r"\brefund\(", path="app/workers", output_mode="content"))

> **Observe:** the definition name finds the definition, the wrapper, a test and a dead legacy copy, which would make the function look barely used. The exported name finds the API route, the support macro (`billing.issue_refund`, a module-qualified call) and the nightly worker, which renames it a second time to `refund`.

> **Try it:** extend `exported_names` to read `__all__` and parenthesized multi-line imports (`from .x import (\n    a as b,\n    c,\n)`). You will need it in the homework.

## 13.8 Built-in tools vs MCP tools

Agents tend to pick built-in tools like Grep over an MCP tool that looks similar, unless the MCP tool's description says what it offers that Grep cannot. The toy scorer below counts shared words between a request and each description, plus a small bonus for built-in tools to model that preference (the CCAR-F exam guide describes it in task 2.4; the Claude Code docs do not state it as a rule). A real model reads descriptions far more cleverly, but the lesson is the same: **a vague description gives the model no reason to choose the tool.** [Theory](README.md#138-built-in-tools-vs-mcp-tools)

In [ ]:
# (offline) A toy description matcher (word overlap); not how Claude chooses, but it shows why descriptions matter
STOP = {"the", "a", "an", "of", "to", "in", "and", "or", "for", "with", "by", "is", "are", "this", "that", "use",
        "which", "what", "where", "who", "it", "on", "across", "all", "from", "when", "instead", "not"}


def words(text: str) -> set[str]:
    return {w for w in re.findall(r"[a-z]+", text.lower()) if w not in STOP and len(w) > 2}


BUILTIN_BIAS = 2   # models the preference for familiar built-in tools that the CCAR-F guide (task 2.4) describes


def pick(request: str, tools: dict[str, str]) -> str:
    scores = {name: len(words(request) & words(desc)) + (0 if name.startswith("mcp__") else BUILTIN_BIAS)
              for name, desc in tools.items()}
    return max(scores, key=scores.get) + f"  {scores}"


BUILTIN_GREP = "Search file contents in the local working directory with a regular expression pattern."
WEAK = "Search code."
STRONG = ("Search code across ALL company service repositories, including repositories that are not checked out "
          "locally, using the central code index. Returns repository, path, line and owning team. Prefer this over "
          "Grep for questions that span services or ask which team owns code.")

request = "Search the code that calls the refunds endpoint across our service repositories and tell me which team owns it"
print("weak  :", pick(request, {"Grep": BUILTIN_GREP, "mcp__codeindex__search": WEAK}))
print("strong:", pick(request, {"Grep": BUILTIN_GREP, "mcp__codeindex__search": STRONG}))
local = "Search the local file contents for the regular expression submit_refund"
print("local :", pick(local, {"Grep": BUILTIN_GREP, "mcp__codeindex__search": STRONG}))

> **Observe:** with the weak description Grep wins, even though the question is about code the local Grep cannot see. With the strong description the MCP tool wins for cross-repository ownership questions, and Grep still wins for a local exact-match search. You did not have to remove Grep; you had to describe the MCP tool's unique data and when to prefer it. (In Claude Code, keep the decisive part near the start: server instructions and upfront tool descriptions are cut at 4,096 characters by default, descriptions loaded through tool search at 16,384, and tool search matches only the first 4,096.)

## 13.9 Other current built-in tools *(beyond the guide)*

The exam's six tools are a subset. The input shapes below come from the Agent SDK's TypeScript reference (checked 2026-11-06; deprecated `Agent` fields left out); checking a call against them is a quick way to catch a wrong field name in a hook, a test fixture or a transcript replay. [Theory](README.md#139-other-current-built-in-tools-beyond-the-guide)

In [ ]:
# (offline) Input fields per tool: (required, optional)
TOOL_INPUTS = {
    "Glob": ({"pattern"}, {"path"}),
    "Grep": ({"pattern"}, {"path", "glob", "type", "output_mode", "-i", "-o", "-n", "-A", "-B", "-C", "context",
                           "head_limit", "offset", "multiline"}),
    "Read": ({"file_path"}, {"offset", "limit", "pages", "allow_large"}),
    "Write": ({"file_path", "content"}, set()),
    "Edit": ({"file_path", "old_string", "new_string"}, {"replace_all"}),
    "Bash": ({"command"}, {"timeout", "description", "run_in_background", "dangerouslyDisableSandbox"}),
    "NotebookEdit": ({"notebook_path", "new_source"}, {"cell_id", "cell_type", "edit_mode"}),
    "WebFetch": ({"url", "prompt"}, {"offset"}),
    "WebSearch": ({"query"}, {"allowed_domains", "blocked_domains"}),
    "Agent": ({"description", "prompt"}, {"subagent_type", "model", "effort", "run_in_background", "name", "isolation"}),
}


def check_call(tool: str, inp: dict) -> list[str]:
    if tool not in TOOL_INPUTS:
        return [f"unknown tool {tool!r} (names are case-sensitive)"]
    required, optional = TOOL_INPUTS[tool]
    problems = [f"missing {k}" for k in sorted(required - inp.keys())]
    problems += [f"unknown field {k}" for k in sorted(inp.keys() - required - optional)]
    if tool == "WebSearch" and {"allowed_domains", "blocked_domains"} <= inp.keys():
        problems.append("allowed_domains and blocked_domains cannot be combined in one call")
    return problems


examples = [
    ("Agent", {"description": "Map refund flow", "prompt": "Find every caller of issue_refund ...", "subagent_type": "Explore"}),
    ("NotebookEdit", {"notebook_path": "notebooks/refund_analysis.ipynb", "cell_id": "c1", "new_source": "print(1)"}),
    ("WebFetch", {"url": "https://docs.python.org/3/library/re.html"}),
    ("Edit", {"path": "app/x.py", "old": "a", "new": "b"}),
    ("grep", {"pattern": "x"}),
    ("WebSearch", {"query": "flask 3 migration", "allowed_domains": ["flask.palletsprojects.com"], "blocked_domains": ["x.com"]}),
]
for tool, inp in examples:
    print(f"{tool:12} {check_call(tool, inp) or 'OK'}")

## 13.10 Permission rules per tool *(beyond the guide)*

Rules are `Tool` or `Tool(specifier)`, evaluated **deny → ask → allow** (first match wins). `Read(...)` rules cover Read, Grep, Glob and LSP, and a `Read` deny also blocks Edit and Write on that path (not NotebookEdit); `Edit(...)` rules cover Edit, Write and NotebookEdit; `Write(...)`, `Glob(...)` and `Grep(...)` path rules are never consulted. A compound Bash command is decided subcommand by subcommand, and Read/Edit deny rules also cover the files that recognized Bash file commands and redirections name. The evaluator below is a **simplified model** of the documented behavior (no modes, hooks, sandbox or wrappers), good enough to reason about a configuration before you try it in `/permissions`. [Theory](README.md#1310-permission-rules-per-tool-beyond-the-guide)

In [ ]:
# (offline) Simplified permission-rule evaluator
# (offline) A SIMPLIFIED model of Claude Code permission rules, for learning only.
# Modeled on code.claude.com/docs/en/permissions: deny -> ask -> allow, first match wins; bare tool names match
# every use; Bash rules match command text with * wildcards; a compound command is split into subcommands
# and EACH subcommand is decided against the whole rule set (any deny -> deny, any ask -> ask, allow only when
# every subcommand is allowed by some allow rule or is read-only); a command ending in && or || is not split,
# so allow rules do not approve it; Read/Edit DENY rules also apply to the files named by Bash file commands
# that Claude Code recognizes (cat/head/tail/sed read, sed -i and tee write) and to redirection targets
# (< file reads, > file writes), but not to arbitrary subprocesses (python script.py) or unnamed files
# (grep -r . from the parent directory); Read(...) rules cover Read/Grep/Glob/LSP, Edit(...) rules cover Edit/Write/NotebookEdit; a Read deny also
# blocks Edit and Write (not NotebookEdit); an Edit allow also grants Read; Write(...), Glob(...) and Grep(...) path rules
# are never consulted; WebFetch(domain:...) matches the host; mcp__server and tool-name globs match MCP tools.
# Not modeled: modes, hooks, sandbox, wrappers, subshells, symlinks, additional directories, settings
# precedence; the recognized Bash file commands are a small assumed subset of the real list. Use the real `/permissions` UI to confirm behavior.
# Assumptions where the docs are silent (checked 2026-10-03): the docs say only "read-only forms of git", so
# the git subset below is a guess; `Read(./.env)` is documented as "the .env file in the current directory"
# and is modeled as anchored there (no doc example covers nested files, so write `Read(**/.env)` when you mean
# every .env); a bare `Edit` (or `Write`) deny is modeled as matching only that one tool.
import fnmatch
import re
import shlex
from urllib.parse import urlparse

READ_TOOLS = {"Read", "Grep", "Glob", "LSP"}
READ_DENY_ALSO_BLOCKS = {"Edit", "Write"}          # the docs: a Read deny does not cover NotebookEdit
EDIT_TOOLS = {"Edit", "Write", "NotebookEdit"}
READ_ONLY_BASH = ("ls", "cat", "head", "tail", "grep", "find", "wc", "pwd", "echo", "which", "diff", "stat", "du",
                  "git status", "git log", "git diff", "git show")   # documented examples + an assumed git subset
PATH_KEYS = ("file_path", "notebook_path", "path")
SEPARATORS = r"&&|\|\||\|&|;|\||&|\n"           # compound-command separators named in the docs
BASH_READ_CMDS = ("cat", "head", "tail", "sed")      # recognized file commands (doc examples; an assumed subset)
BASH_WRITE_CMDS = ("tee",)


def parse_rule(rule: str) -> tuple[str, str | None]:
    m = re.fullmatch(r"\s*([A-Za-z_*][\w*]*)\s*(?:\((.*)\))?\s*", rule)
    if not m:
        raise ValueError(f"not a permission rule: {rule!r}")
    return m.group(1), m.group(2)


def _bash_match(spec: str, command: str) -> bool:
    spec = spec[:-2] + " *" if spec.endswith(":*") else spec
    rx = "".join(".*" if ch == "*" else re.escape(ch) for ch in spec)
    if spec.endswith(" *") and spec.count("*") == 1 and command == spec[:-2]:
        return True                                   # "npm run *" also matches bare "npm run"
    return re.fullmatch(rx, command) is not None


def _glob_rx(spec: str) -> str:
    i, out = 0, []
    while i < len(spec):
        if spec.startswith("**/", i):
            out.append("(?:.*/)?"); i += 3; continue
        if spec.startswith("/**", i) and i + 3 == len(spec):
            out.append("(?:/.*)?"); i += 3; continue
        if spec.startswith("**", i):
            out.append(".*"); i += 2; continue
        c = spec[i]
        out.append("[^/]*" if c == "*" else "[^/]" if c == "?" else re.escape(c)); i += 1
    return "".join(out)


def _path_match(spec: str, path: str, kind: str) -> bool:
    """gitignore-flavoured, with the project root as the working directory:
    '//x' absolute, '/x' and './x' anchored at the root, a bare name (no '/') at any depth,
    and a single-directory pattern such as 'secrets/**' at any depth for deny/ask rules only."""
    path = path[2:] if path.startswith("./") else path.lstrip("/")
    anchored = spec.startswith(("/", "./"))
    spec = spec[2:] if spec.startswith("//") else spec[2:] if spec.startswith("./") else spec.lstrip("/")
    body = spec[:-3] if spec.endswith("/**") else spec
    any_depth = not anchored and ("/" not in spec or (kind != "allow" and "/" not in body))
    return re.fullmatch(("(?:.*/)?" if any_depth else "") + _glob_rx(spec), path) is not None


def rule_matches(rule: str, tool: str, inp: dict, kind: str) -> bool:
    name, spec = parse_rule(rule)
    if name in ("Write", "Glob", "Grep", "NotebookEdit", "MultiEdit") and spec:   # never consulted for paths
        return False
    if spec in (None, "", "*"):
        if name == tool or (name.startswith("mcp__") and "*" not in name and tool.startswith(name + "__")):
            return True                                   # a bare tool name, or every tool of one MCP server
        if "*" in name:                                   # tool-name globs: "*", "mcp__*" (deny/ask only);
            anchored = re.match(r"mcp__[^*]+?__", name)   # allow globs need a literal mcp__<server>__ prefix
            return (kind != "allow" or anchored is not None) and fnmatch.fnmatchcase(tool, name)
        return False
    if name == "Bash" and tool in ("Bash", "Monitor"):
        return _bash_match(spec, inp.get("command", "").strip())      # one subcommand (decide() splits)
    if name == "WebFetch" and tool == "WebFetch" and spec.startswith("domain:"):
        host, pat = (urlparse(inp.get("url", "")).hostname or "").lower(), spec[len("domain:"):].lower()   # case-insensitive
        if pat == "*":
            return True
        if pat.startswith("*."):
            return host.endswith(pat[1:])
        return host == pat
    if name == "Agent" and tool == "Agent":
        return inp.get("subagent_type") == spec
    path = next((inp[k] for k in PATH_KEYS if inp.get(k)), None)
    if path is None:
        return False
    if name == "Read" and (tool in READ_TOOLS or (kind == "deny" and tool in READ_DENY_ALSO_BLOCKS)):
        return _path_match(spec, path, kind)
    if name == "Edit" and (tool in EDIT_TOOLS or (kind == "allow" and tool in READ_TOOLS)):
        return _path_match(spec, path, kind)
    return False


def bash_parts(command: str) -> list[str]:
    return [p.strip() for p in re.split(SEPARATORS, command) if p.strip()]


def bash_file_targets(part: str) -> list[tuple[str, str]]:
    """(tool, path) pairs for the files one subcommand names: ('Read', p) for reads, ('Edit', p) for writes."""
    out = []
    for op, target in re.findall(r"(?:^|\s)\d?(>>|>|<)\s*([^\s&|;<>]+)", part):    # redirection targets
        if not target.startswith("&") and target != "/dev/null":
            out.append(("Read" if op == "<" else "Edit", target))
    words = re.sub(r"(?:^|\s)\d?(>>|>|<)\s*[^\s&|;<>]+", " ", part)
    try:
        tokens = shlex.split(words)
    except ValueError:
        tokens = words.split()
    if not tokens:
        return out
    cmd, args = tokens[0], tokens[1:]
    operands = [t for t in args if not t.startswith("-") and not t.isdigit()]
    if cmd == "sed":
        in_place = any(t == "--in-place" or re.match(r"-[a-zA-Z]*i", t) for t in args)
        if not any(t in ("-e", "--expression") or t.startswith("-e") for t in args) and operands:
            operands = operands[1:]                                    # the first operand is the sed script
        out += [("Read", f) for f in operands] + ([("Edit", f) for f in operands] if in_place else [])
    elif cmd in BASH_READ_CMDS:
        out += [("Read", f) for f in operands]
    elif cmd in BASH_WRITE_CMDS:
        out += [("Edit", f) for f in operands]
    return out


def _decide_bash_part(rules: dict, tool: str, part: str, allow_ok: bool) -> str:
    inp = {"command": part}
    if any(rule_matches(r, tool, inp, "deny") for r in rules.get("deny", [])):
        return "deny"
    if tool == "Bash":                                  # Read/Edit deny rules also cover recognized file commands
        for file_tool, path in bash_file_targets(part):
            if any(parse_rule(r)[0] in ("Read", "Edit") and rule_matches(r, file_tool, {"file_path": path}, "deny")
                   for r in rules.get("deny", [])):
                return "deny"
    if any(rule_matches(r, tool, inp, "ask") for r in rules.get("ask", [])):
        return "ask"
    if allow_ok and any(rule_matches(r, tool, inp, "allow") for r in rules.get("allow", [])):
        return "allow"
    if tool == "Bash" and any(part == c or part.startswith(c + " ") for c in READ_ONLY_BASH):
        return "allow"                                  # built-in read-only command
    return "ask"


def default_decision(tool: str, inp: dict) -> str:
    if tool in READ_TOOLS | {"Agent"}:
        return "allow"                                  # inside the working directory
    return "ask"                                        # edits, writes, web fetch/search, other tools


def decide(rules: dict, tool: str, inp: dict) -> str:
    """Return 'deny', 'ask' or 'allow' for one tool call under {'allow': [...], 'ask': [...], 'deny': [...]}."""
    if tool in ("Bash", "Monitor"):
        command = inp.get("command", "")
        parts = bash_parts(command)
        if not parts:
            return "ask"
        allow_ok = re.search(r"(&&|\|\|)\s*$", command) is None      # "npm test &&" is unparseable for allow rules
        decisions = [_decide_bash_part(rules, tool, p, allow_ok) for p in parts]
        return "deny" if "deny" in decisions else "ask" if "ask" in decisions else "allow"
    for kind in ("deny", "ask", "allow"):
        for rule in rules.get(kind, []):
            if rule_matches(rule, tool, inp, kind):
                return kind
    return default_decision(tool, inp)


def lint_rules(rules: dict) -> list[str]:
    """Warnings Claude Code would also give (or that this course recommends)."""
    warnings = []
    for kind, items in rules.items():
        for r in items:
            name, spec = parse_rule(r)
            if name in ("Write", "Glob", "Grep", "NotebookEdit", "MultiEdit") and spec:
                warnings.append(f"{kind}: {r} is never consulted for paths; use "
                                f"{'Read' if name in ('Glob', 'Grep') else 'Edit'}({spec})")
            if kind == "allow" and name == "Bash" and spec and re.match(r"^\S+ \*", spec) and spec.split()[0] in ("git", "npm", "docker", "kubectl"):
                warnings.append(f"allow: {r} puts * right after the program name and allows every subcommand")
    return warnings

In [ ]:
# (offline) A team configuration, checked against concrete tool calls
TEAM_RULES = {
    "allow": ["Bash(npm run test *)", "Bash(npm run lint)", "WebFetch(domain:docs.python.org)"],
    "ask": ["Bash(git push *)"],
    "deny": ["Read(./.env)", "Read(secrets/**)", "Edit(/migrations/**)", "Bash(curl *)"],
}
CALLS = [
    ("Read", {"file_path": ".env"}), ("Edit", {"file_path": ".env"}), ("Grep", {"pattern": "KEY", "path": "secrets"}),
    ("Write", {"file_path": "migrations/0042_refunds.sql"}), ("NotebookEdit", {"notebook_path": "migrations/x.ipynb"}),
    ("Edit", {"file_path": "app/billing/policy.py"}), ("Read", {"file_path": "app/billing/policy.py"}),
    ("Bash", {"command": "npm run test -- --run"}), ("Bash", {"command": "npm install left-pad"}),
    ("Bash", {"command": "npm run lint && curl https://example.com"}), ("Bash", {"command": "npm run test && npm run lint"}),
    ("Bash", {"command": "cat .env"}), ("Bash", {"command": "python scripts/load_env.py"}), ("Bash", {"command": "git log -5"}),
    ("Bash", {"command": "git push origin main"}), ("WebFetch", {"url": "https://docs.python.org/3/"}),
    ("WebFetch", {"url": "https://pastebin.example/raw/1"}),
]
for tool, inp in CALLS:
    print(f"{decide(TEAM_RULES, tool, inp):5}  {tool}({next(iter(inp.values()))})")
print("\nlint:", lint_rules({"deny": ["Write(./infra/**)"], "allow": ["Bash(git *)"]}))

> **Observe:** `npm run lint && curl ...` is denied even though the first half is allowed: each subcommand is checked on its own, and any deny wins. `npm run test && npm run lint` is allowed because each subcommand matches *some* allow rule; one rule does not have to cover both. `cat .env` is denied by the `Read(./.env)` rule because Claude Code applies Read and Edit deny rules to the files that recognized Bash file commands (`cat`, `head`, `tail`, `sed`, `tee`) and redirections name. `python scripts/load_env.py` only asks: a script that opens files itself is not covered by path rules, which is why real boundaries need the sandbox. `git log -5` needs no rule because it is in the built-in read-only set. The `Write(./infra/**)` rule would silently do nothing.

**The same names configure an Agent SDK agent.** Building the options object is offline; nothing starts until you call `query()`:

In [ ]:
# (offline) Agent SDK options for a read-only investigator
from claude_agent_sdk import ClaudeAgentOptions

investigator = ClaudeAgentOptions(
    model=MODEL,
    tools=["Read", "Grep", "Glob"],            # the ONLY built-in tools this agent has (also restores Grep/Glob on macOS/Linux)
    allowed_tools=["Read", "Grep", "Glob"],    # pre-approved: no prompts for these
    permission_mode="dontAsk",                 # anything not pre-approved is denied instead of prompting
    setting_sources=[],                        # ignore personal ~/.claude settings and CLAUDE.md files
    max_turns=15,
    max_budget_usd=0.30,
)
print({k: getattr(investigator, k) for k in ("tools", "allowed_tools", "disallowed_tools", "permission_mode", "max_budget_usd")})

risky = ClaudeAgentOptions(allowed_tools=["Read"], permission_mode="bypassPermissions")
print("\nrisky config: allowed_tools =", risky.allowed_tools, "| tools =", risky.tools, "| mode =", risky.permission_mode)
print("-> every other tool (Bash, Write, Edit ...) is still present AND approved by bypassPermissions")

## 13.11 Codebase understanding for modernization work

Before a modernization plan, measure where the risk is: **fan-in** (how many files depend on a module), **churn** (how often it changes) and **test gaps**. All three come from the built-in tools: Grep counts, `git log` through Bash, and Glob for test files. [Theory](README.md#1311-codebase-understanding-for-modernization-work)

In [ ]:
# (offline) Hotspot ranking from tool evidence
repo = new_repo()
churn = {}
for row in repo.call("Bash", command="git log --name-only --since=12.months").output.splitlines():
    n, _, path = row.split(maxsplit=2)
    churn[path] = int(n)

modules = [f for f in repo.call("Glob", pattern="app/**/*.py").output.splitlines()
           if f.endswith(".py") and not f.endswith("__init__.py") and "/generated/" not in f]
rows = []
for m in modules:
    stem = Path(m).stem
    dotted = m[:-3].replace("/", ".")
    fan_in_res = repo.call("Grep", pattern=rf"(from|import)\s+({re.escape(dotted)}\b|\.{stem}\b)", type="py")
    fan_in = len([f for f in fan_in_res.output.splitlines() if f.endswith(".py") and f != m])
    has_tests = repo.call("Glob", pattern=f"tests/**/test_{stem}.py").output != "No files found"
    score = (fan_in + 1) * (1 + churn.get(m, 0)) * (1 if has_tests else 2)
    rows.append((score, m, fan_in, churn.get(m, 0), has_tests))

print(f"{'score':>6}  {'module':28} fan-in churn tests")
for score, m, fan_in, c, t in sorted(rows, reverse=True):
    print(f"{score:>6}  {m:28} {fan_in:>6} {c:>5}  {'yes' if t else 'NO'}")
print(f"\n{len(repo.ledger)} tool calls, ~{repo.context_tokens():,} tokens, 0 files read in full")

> **Observe:** `refunds.py` and `gateway.py` rise to the top: they change most often, `refunds.py` has more dependents than most modules, and the gateway has no tests. (`ledger.py` has the highest fan-in, but it changes less and already has tests.) That is evidence for where a "add tests to the legacy code" engagement should start, gathered without reading a single file in full. The next steps are a plan in plan mode, then slices of work, adjusting the plan as dependencies appear (for example, the nightly worker calls the refund function by a second alias, `refund`, so the plan must cover that caller too).

> **Try it:** change the score so that modules touching money (`billing`, `payments`) get a risk multiplier. Does the order change?

## Mini-project: an offline investigation agent

`InvestigationAgent` answers *"Which public entry points can lead to this error?"* using only Grep, Glob and Read, following the chapter's rules:

1. Grep the error text (content mode) → the raising file and line.
2. Read that file once → the enclosing function.
3. Grep for calls to that function → the next files up the chain; Read each once to find *their* enclosing function.
4. When a function is re-exported by a wrapper module (`import x as y`), switch to searching for the exported name.
5. Stop at functions decorated with `@route` (entry points) or when nothing calls the function; record every step in a scratchpad.

In [ ]:
# (offline) The agent
DEF_RX = re.compile(r"^\s*def (\w+)\(", re.M)


class InvestigationAgent:
    def __init__(self, repo: FakeRepo, max_reads: int = 8):
        self.repo, self.max_reads, self.reads, self.notes = repo, max_reads, {}, []

    def read(self, path: str) -> list[str]:
        if path not in self.reads:
            if len(self.reads) >= self.max_reads:
                raise RuntimeError("read budget exhausted")
            self.reads[path] = (self.repo.read_raw(path) or "").splitlines()
        return self.reads[path]

    def enclosing_function(self, path: str, line_no: int) -> tuple[str | None, bool]:
        lines = self.read(path)
        for i in range(line_no - 1, -1, -1):
            m = re.match(r"\s*def (\w+)\(", lines[i])
            if m:
                is_route = i > 0 and lines[i - 1].lstrip().startswith("@route")
                return m.group(1), is_route
        return None, False

    def aliases(self, name: str) -> list[str]:
        """All names the function is imported under ('name as alias'), followed transitively (alias of an alias)."""
        found, todo = [], [name]
        while todo:
            current = todo.pop()
            res = self.repo.call("Grep", pattern=rf"\b{current} as (\w+)", type="py", output_mode="content")
            for alias in sorted({m for ln in res.output.splitlines() for m in re.findall(rf"\b{current} as (\w+)", ln)}):
                if alias not in found and alias != name:
                    found.append(alias)
                    todo.append(alias)
        return found

    def callers(self, name: str, defined_in: str) -> list[tuple[str, int]]:
        res = self.repo.call("Grep", pattern=rf"\b{name}\(", type="py", output_mode="content")
        out = []
        for ln in res.output.splitlines():
            m = re.match(r"([^:]+):(\d+):(.*)", ln)
            if m and m.group(1) != defined_in and not m.group(1).startswith(("tests/", "legacy/")) \
                    and not re.match(r"\s*def ", m.group(3)):
                out.append((m.group(1), int(m.group(2))))
        return out

    def investigate(self, error_text: str) -> list[list[str]]:
        hit = self.repo.call("Grep", pattern=re.escape(error_text), output_mode="content").output.splitlines()
        path, line, _ = hit[0].split(":", 2)
        fn, _ = self.enclosing_function(path, int(line))
        self.notes.append(f"- `{error_text}` raised in {path}:{line} inside `{fn}`")
        chains, frontier = [], [([f"{path}:{line} {fn}"], fn, path)]
        while frontier:
            chain, name, defined_in = frontier.pop()
            names = [name] + self.aliases(name)
            found = False
            for n in names:
                for cpath, cline in self.callers(n, defined_in):
                    cfn, is_route = self.enclosing_function(cpath, cline)
                    step = f"{cpath}:{cline} {cfn}" + (" [ROUTE]" if is_route else "") + (f" (calls it as {n})" if n != name else "")
                    found = True
                    self.notes.append(f"- {cfn} in {cpath}:{cline} calls `{n}`" + (" (entry point)" if is_route else ""))
                    if is_route or cfn is None:
                        chains.append([step] + chain)
                    else:
                        frontier.append(([step] + chain, cfn, cpath))
            if not found:
                chains.append(chain)
        return chains


repo = new_repo()
agent = InvestigationAgent(repo)
chains = agent.investigate("Refund declined by processor")
for c in sorted(chains):
    print("  ->  ".join(c))
scratch = "# investigation-scratchpad.md\n## Confirmed findings\n" + "\n".join(agent.notes) + "\n"
repo.call("Write", file_path="notes/investigation-scratchpad.md", content=scratch)
print("\n" + scratch)
print(f"{len(repo.ledger)} tool calls: {dict(Counter(r.tool for r in repo.ledger))}, files read: {len(agent.reads)}, "
      f"~{repo.context_tokens():,} tokens")

In [ ]:
# (offline) Same question, "read everything first" baseline
baseline = new_repo()
for f in baseline.call("Glob", pattern="**/*.py").output.splitlines():
    if f.endswith(".py"):
        baseline.call("Read", file_path=f)
print(f"read-all baseline: {len(baseline.ledger)} calls, ~{baseline.context_tokens():,} tokens "
      f"vs the agent's ~{repo.context_tokens():,}")

> **Observe:** the agent found the HTTP route, the support macro and the nightly retry job (which calls the function by a *second* alias, `refund`). It read only the files on the chain, and every finding in the scratchpad has a file and line.

> **Try it:** run `InvestigationAgent(new_repo()).investigate("Ledger entry rejected: account closed")`. Which entry points can post a ledger entry for a closed account, and how many reads did it take?

### (live, optional) The same task with the Claude Agent SDK

This cell copies the repository into a temporary directory (without the gitignored `build/` folder) and runs a real agent restricted to `Read`, `Grep` and `Glob`, in `dontAsk` mode, with `max_turns` and a `max_budget_usd` stopping threshold (based on the SDK's client-side cost estimate, so not a hard spending cap). It never touches your working copy. It runs only when `RUN_LIVE = True`. This cell was checked statically against `claude-agent-sdk` 0.2.163 (option and message field names) but has not been run end to end, so your tool trace and cost may differ from the offline mini-project.

In [ ]:
# (live, optional) Requires RUN_LIVE = True and ANTHROPIC_API_KEY; stops near an estimated $0.30 (can overshoot slightly)
if not RUN_LIVE:
    print("Skipped: set RUN_LIVE = True in the setup cell (and ANTHROPIC_API_KEY in .env) to run this cell.")
else:
    require_live()
    from claude_agent_sdk import AssistantMessage, ResultMessage, TextBlock, ToolUseBlock, query

    workdir = Path(tempfile.mkdtemp(prefix="ch13-acme-"))
    for rel, text in ACME_FILES.items():
        if rel.startswith("build/"):
            continue
        target = workdir / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(text)

    live_options = ClaudeAgentOptions(
        model=MODEL, cwd=str(workdir), tools=["Read", "Grep", "Glob"], allowed_tools=["Read", "Grep", "Glob"],
        permission_mode="dontAsk", setting_sources=[], max_turns=15, max_budget_usd=0.30,
    )
    prompt = ("Customers see the error 'Refund declined by processor'. Trace every code path from a public entry point "
              "(HTTP route, support macro, background job) to the line that raises it. Watch for wrapper modules that "
              "re-export functions under other names. Investigate incrementally: search first, read only what you need. "
              "Answer with one chain per entry point as file:line steps.")
    async for message in query(prompt=prompt, options=live_options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, ToolUseBlock):
                    print("tool_use ->", block.name, json.dumps(block.input)[:120])
                elif isinstance(block, TextBlock):
                    print(block.text)
        elif isinstance(message, ResultMessage):
            cost = f"${message.total_cost_usd:.4f}" if message.total_cost_usd is not None else "n/a"
            print(f"RESULT {message.subtype} | turns={message.num_turns} | cost={cost}")

> **Observe (live):** compare the real agent's tool sequence with the offline agent's ledger. Did it start with a search? Did it find the `refund` alias in the nightly worker? How many files did it read?

## Recap

- **Paths → Glob, contents → Grep, known file → Read**, new file or full rewrite → Write, targeted change → Edit, running things → Bash. Watch the Glob cap and the `.gitignore` difference; escape regex metacharacters in Grep.
- **Edit = read first + exact match + unique match** (or `replace_all`). Failures are deterministic: diagnose, widen the anchor, and when no small unique anchor exists, **Read the full file, modify it in code, Write it back, verify**.
- **Investigate incrementally**: Grep the entry point, Read it, Grep what it calls, Read those; stop when answered; keep a scratchpad of file:line findings.
- **Wrapper modules hide callers**: list every exported name and alias, and search for each one.
- **MCP tools win on description**, not by removing built-ins.
- **Permissions use the same tool names**: deny → ask → allow; `Read(...)` and `Edit(...)` are the path rules; in the Agent SDK, `tools` sets the base set, `allowed_tools` only pre-approves.
- **Modernization starts with evidence**: fan-in, churn and test gaps from Grep, `git log` and Glob, then a plan.

Next: [homework](02_homework.ipynb)